# 15 — Delta Spatiotemporal Forecasting

Bu notebook hedef değişkeni ve problem formülasyonunu yeniden ele alır. Referans, public-best `11_submission_optimization.ipynb` sistemidir. Bütün karar eşikleri sonuçlar görülmeden önce sabitlenmiştir.

**Leakage sözleşmesi**

- Üç expanding-window temporal fold kullanılır.
- Validation/test target hiçbir feature'a girmez.
- Target-türevli lag, climatology, trend ve spatial özetler yalnız fold training geçmişinden üretilir.
- Validation içinde teacher forcing yapılmaz; recursive deney yalnız kendi tahminini ileri taşır.
- Public skor katsayı veya model seçmekte kullanılmaz.
- `14`teki local ensemble public'e taşınmadığı için seçim yalnız ortalama RMSE değil robust score ve worst-fold ile yapılır.


In [1]:
from pathlib import Path
from time import perf_counter
import gc, warnings, pickle
import numpy as np, pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE=True
except Exception:
    CATBOOST_AVAILABLE=False
warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED)
ROOT=Path.cwd().resolve()
if not (ROOT/"data/raw/Train.csv").exists(): ROOT=ROOT.parent
DATA=ROOT/"data/raw"; OUT=ROOT/"submissions"; OUT.mkdir(exist_ok=True)
ROBUST_STD_WEIGHT=.25
ROBUST_WORST_WEIGHT=.25
MIN_OVERALL_PRIMARY=.0015
ALT_OVERALL=.0008
ALT_WORST=.002
MASKED_GAIN=.004
CORR_LIMIT=.995
print(ROOT, CATBOOST_AVAILABLE)


C:\global-water-storage-forecasting True


## 1. Veri, feature şeması ve sabit temporal validation

In [2]:
train=pd.read_csv(DATA/"Train.csv",parse_dates=["time"])
test=pd.read_csv(DATA/"Test.csv",parse_dates=["time"])
sample=pd.read_csv(DATA/"SampleSubmission.csv")
SPEI=["SPEI_01_t","SPEI_03_t","SPEI_06_t","SPEI_12_t"]
def base_features(df):
    x=df.copy()
    x["year"]=x.time.dt.year.astype("int16"); x["month"]=x.time.dt.month.astype("int8"); x["quarter"]=x.time.dt.quarter.astype("int8")
    x["elapsed_month"]=((x.time.dt.year-train.time.dt.year.min())*12+x.time.dt.month-1).astype("int16")
    x["sin_month"]=np.sin(2*np.pi*x.month/12)
    x["cos_month"]=np.cos(2*np.pi*x.month/12)
    x["season_sin"]=np.sin(2*np.pi*x.quarter/4); x["season_cos"]=np.cos(2*np.pi*x.quarter/4)
    s=x[SPEI]; x["spei_mean"]=s.mean(axis=1); x["spei_min"]=s.min(axis=1); x["spei_max"]=s.max(axis=1); x["spei_std"]=s.std(axis=1)
    x["spei_short_long"]=(x.SPEI_01_t-x.SPEI_12_t).astype("float32")
    x["spei_03_minus_12"]=(x.SPEI_03_t-x.SPEI_12_t).astype("float32")
    x["spei_short_long_ratio"]=x.SPEI_01_t/(x.SPEI_12_t.abs()+.1)
    x["abs_lat"]=x.lat.abs().astype("float32")
    x["lat_band"]=(np.floor((x.lat+90)/5)).astype("int16"); x["lon_band"]=(np.floor((x.lon+180)/5)).astype("int16")
    x["spatial_bin_10"]=x.lat_band*100+x.lon_band; x["lat_lon_interaction"]=x.lat*x.lon/1000
    x["lat_sin"]=np.sin(np.radians(x.lat)); x["lat_cos"]=np.cos(np.radians(x.lat))
    x["lon_sin"]=np.sin(np.radians(x.lon)); x["lon_cos"]=np.cos(np.radians(x.lon))
    x["hemisphere"]=(x.lat>=0).astype("int8")
    # Masked uzmanında train ve inference routing sabit olarak 1'dir; known uzman bu feature'ı kullanmaz.
    x["tws_missing"]=np.int8(1)
    return x
train_f=base_features(train); test_f=base_features(test)
BASE=["lat","lon",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year","quarter","elapsed_month",
      "sin_month","cos_month","season_sin","season_cos","spei_mean","spei_min","spei_max","spei_std",
      "spei_short_long","spei_03_minus_12","spei_short_long_ratio","lat_band","lon_band","spatial_bin_10",
      "lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
KNOWN_FEATURES=["lat","lon","TWS_t",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year","quarter",
      "elapsed_month","sin_month","cos_month","season_sin","season_cos","spei_mean","spei_min","spei_max",
      "spei_std","spei_short_long","spei_03_minus_12","spei_short_long_ratio","lat_band","lon_band",
      "spatial_bin_10","lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
MASKED_BASE=BASE
MASKED_FEATURES=BASE+["tws_missing"]
date_stats=test.groupby("time").agg(rows=("ID","size"),known=("TWS_t","count"))
KNOWN_POS=set(np.flatnonzero(date_stats.known.to_numpy()>date_stats.rows.to_numpy()*.5))
dates=np.array(sorted(train_f.time.unique())); start=len(dates)-3*18
FOLDS=[]
for i in range(3):
    vd=dates[start+i*18:start+(i+1)*18]; td=dates[:start+i*18]
    tr=train_f[train_f.time.isin(td)].copy(); va=train_f[train_f.time.isin(vd)].copy()
    vdates=np.array(sorted(va.time.unique()))
    known=va.time.isin({d for j,d in enumerate(vdates) if j in KNOWN_POS}).to_numpy()
    FOLDS.append(dict(fold=i+1,train=tr,valid=va,known=known))
    print(i+1,td.min(),td.max(),vd.min(),vd.max(),len(tr),len(va),known.sum(),(~known).sum())
print("Feature lists\nKNOWN:",KNOWN_FEATURES,"\nMASKED:",MASKED_FEATURES)


1 2002-05-01 00:00:00 2009-09-01 00:00:00 2009-10-01 00:00:00 2011-05-01 00:00:00 1311187 280836 93775 187061


2 2002-05-01 00:00:00 2011-05-01 00:00:00 2011-08-01 00:00:00 2013-07-01 00:00:00 1592023 280936 93511 187425


3 2002-05-01 00:00:00 2013-07-01 00:00:00 2013-11-01 00:00:00 2015-08-01 00:00:00 1872959 281062 93638 187424
Feature lists
KNOWN: ['lat', 'lon', 'TWS_t', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month', 'month_sin', 'month_cos', 'year', 'quarter', 'elapsed_month', 'sin_month', 'cos_month', 'season_sin', 'season_cos', 'spei_mean', 'spei_min', 'spei_max', 'spei_std', 'spei_short_long', 'spei_03_minus_12', 'spei_short_long_ratio', 'lat_band', 'lon_band', 'spatial_bin_10', 'lat_lon_interaction', 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos'] 
MASKED: ['lat', 'lon', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month', 'month_sin', 'month_cos', 'year', 'quarter', 'elapsed_month', 'sin_month', 'cos_month', 'season_sin', 'season_cos', 'spei_mean', 'spei_min', 'spei_max', 'spei_std', 'spei_short_long', 'spei_03_minus_12', 'spei_short_long_ratio', 'lat_band', 'lon_band', 'spatial_bin_10', 'lat_lon_interaction', 'lat_sin', 'lat_cos', 'lon_si

## 2. Causal history ve spatial state

Training satırlarında `shift(1)` sonrası expanding/rolling kullanılır. Validation ve testte yalnız training döneminin son state'i uygulanır. Spatial çözünürlükler 20°, 10° ve 5° olarak önceden sabittir.


In [3]:
BASE_HIST=["hist_loc_mean","hist_loc_std","hist_loc_month","hist_global_month"]
HIST=BASE_HIST+["hist_loc_median",
      "hist_delta_mean","hist_delta_std","hist_delta_median","hist_delta_roll3","hist_delta_roll6",
      "hist_delta_roll12","months_since","hist_trend"]
def causal_history(df):
    o=df.sort_values(["lat","lon","time"]).copy()
    g=o.groupby(["lat","lon"],sort=False)
    prev=g.target.shift(1); delta=o.target-o.TWS_t
    pdelta=delta.groupby([o.lat,o.lon]).shift(1)
    o["hist_loc_mean"]=prev.groupby([o.lat,o.lon]).expanding().mean().reset_index(level=[0,1],drop=True)
    o["hist_loc_std"]=prev.groupby([o.lat,o.lon]).expanding().std().reset_index(level=[0,1],drop=True)
    o["hist_loc_median"]=prev.groupby([o.lat,o.lon]).expanding().median().reset_index(level=[0,1],drop=True)
    o["hist_loc_month"]=prev.groupby([o.lat,o.lon,o.month]).expanding().mean().reset_index(level=[0,1,2],drop=True)
    o["hist_global_month"]=o.groupby("month").target.transform(lambda z:z.shift(1).expanding().mean())
    o["hist_delta_mean"]=pdelta.groupby([o.lat,o.lon]).expanding().mean().reset_index(level=[0,1],drop=True)
    o["hist_delta_std"]=pdelta.groupby([o.lat,o.lon]).expanding().std().reset_index(level=[0,1],drop=True)
    o["hist_delta_median"]=pdelta.groupby([o.lat,o.lon]).expanding().median().reset_index(level=[0,1],drop=True)
    for w in [3,6,12]:
        o[f"hist_delta_roll{w}"]=pdelta.groupby([o.lat,o.lon]).rolling(w,min_periods=1).mean().reset_index(level=[0,1],drop=True)
    prev_time=g.time.shift(1); o["months_since"]=((o.time.dt.year-prev_time.dt.year)*12+o.time.dt.month-prev_time.dt.month).astype("float32")
    prev2=g.target.shift(2); o["hist_trend"]=prev-prev2
    return o.sort_index()
def history_state_apply(h,a):
    loc=h.groupby(["lat","lon"]).target.agg(hist_loc_mean="mean",hist_loc_std="std",hist_loc_median="median")
    lm=h.groupby(["lat","lon","month"]).target.agg(["sum","count"]).rename(columns={"sum":"lm_sum","count":"lm_count"})
    gm=h.groupby("month").target.mean().rename("hist_global_month")
    d=(h.target-h.TWS_t).groupby([h.lat,h.lon]).agg(hist_delta_mean="mean",hist_delta_std="std",hist_delta_median="median")
    last=h.sort_values("time").groupby(["lat","lon"]).tail(2)
    last1=last.groupby(["lat","lon"]).tail(1).set_index(["lat","lon"])[["time","target"]]
    trend=last.groupby(["lat","lon"]).target.agg(lambda z:z.iloc[-1]-z.iloc[-2] if len(z)>1 else 0).rename("hist_trend")
    o=a.join(loc,on=["lat","lon"]).join(lm,on=["lat","lon","month"]).join(gm,on="month").join(d,on=["lat","lon"]).join(trend,on=["lat","lon"])
    o["hist_loc_month"]=(o.lm_sum+4*o.hist_loc_mean)/(o.lm_count.fillna(0)+4)
    o=o.drop(columns=["lm_sum","lm_count"])
    for w in [3,6,12]:
        roll=h.sort_values("time").groupby(["lat","lon"]).tail(w).groupby(["lat","lon"]).apply(lambda z:(z.target-z.TWS_t).mean(),include_groups=False).rename(f"hist_delta_roll{w}")
        o=o.join(roll,on=["lat","lon"])
    lt=o.set_index(["lat","lon"]).index.map(last1.time)
    o["months_since"]=((o.time.dt.year-pd.DatetimeIndex(lt).year)*12+o.time.dt.month-pd.DatetimeIndex(lt).month).astype("float32")
    return o
def add_spatial(df,res):
    o=df.copy(); o[f"cell_{res}_lat"]=np.floor((o.lat+90)/res).astype("int16"); o[f"cell_{res}_lon"]=np.floor((o.lon+180)/res).astype("int16")
    return o
def causal_spatial_train(df,res):
    o=add_spatial(df,res); keys=[f"cell_{res}_lat",f"cell_{res}_lon","month"]
    o[f"cell{res}_month_mean"]=o.groupby(keys).target.transform(lambda z:z.shift(1).expanding().mean())
    return o
def apply_spatial(h,a,res):
    hh=add_spatial(h,res); o=add_spatial(a,res); keys=[f"cell_{res}_lat",f"cell_{res}_lon","month"]
    stat=hh.groupby(keys).target.mean().rename(f"cell{res}_month_mean")
    return o.join(stat,on=keys)
for f in FOLDS:
    f["train_hist"]=causal_history(f["train"]); f["valid_hist"]=history_state_apply(f["train"],f["valid"])
gc.collect()


0

## 3. Ortak OOF ölçümü ve Notebook 11 baseline reproduction

In [4]:
LGB_PARAMS=dict(n_estimators=160,learning_rate=.08,num_leaves=63,min_child_samples=100,
    reg_lambda=1.0,subsample=.85,colsample_bytree=.9,random_state=SEED,n_jobs=4,verbosity=-1)
def lgb(): return LGBMRegressor(**LGB_PARAMS)
def frame(f,p,mask,segment,name):
    o=f["valid"].loc[mask,["sample_id","time","lat","lon","target",*SPEI,"SOIL_MOISTURE_t","TWS_t"]].copy()
    o["prediction"]=np.asarray(p); o["segment"]=segment; o["fold"]=f["fold"]; o["model"]=name
    return o
def summarize(o,name):
    folds=o.groupby("fold").apply(lambda z:root_mean_squared_error(z.target,z.prediction),include_groups=False)
    k=o[o.segment=="known"]; m=o[o.segment=="masked"]
    q10=train_f.SPEI_03_t.quantile(.1); q90=train_f.SPEI_03_t.quantile(.9)
    dry=o[o.SPEI_03_t<=q10]; wet=o[o.SPEI_03_t>=q90]; recent=o[o.time>=o.time.quantile(.75)]
    mean=root_mean_squared_error(o.target,o.prediction); worst=folds.max(); std=folds.std(ddof=0)
    return {"Model":name,"Overall RMSE":mean,"Known RMSE":root_mean_squared_error(k.target,k.prediction) if len(k) else np.nan,
      "Masked RMSE":root_mean_squared_error(m.target,m.prediction) if len(m) else np.nan,"MAE":mean_absolute_error(o.target,o.prediction),
      "Worst-fold RMSE":worst,"Fold std":std,"Dry RMSE":root_mean_squared_error(dry.target,dry.prediction),
      "Wet RMSE":root_mean_squared_error(wet.target,wet.prediction),"Recent RMSE":root_mean_squared_error(recent.target,recent.prediction),
      "Robust score":mean+ROBUST_STD_WEIGHT*std+ROBUST_WORST_WEIGHT*max(0,worst-mean),"OOF rows":len(o)}
EXPS=[]
def add(name,o,kind="model",meta=None):
    r={"name":name,"oof":o,"metrics":summarize(o,name),"kind":kind,"meta":meta or {}}; EXPS.append(r); return r
known_parts=[]; masked_base=[]; masked_hist=[]
for f in FOLDS:
    k=lgb().fit(f["train"][KNOWN_FEATURES],f["train"].target)
    known_parts.append(frame(f,k.predict(f["valid"].loc[f["known"],KNOWN_FEATURES]),f["known"],"known","11"))
    m=lgb().fit(f["train"][MASKED_BASE],f["train"].target)
    masked_base.append(frame(f,m.predict(f["valid"].loc[~f["known"],MASKED_BASE]),~f["known"],"masked","11"))
    h=lgb().fit(f["train_hist"][MASKED_BASE+BASE_HIST],f["train_hist"].target)
    masked_hist.append(frame(f,h.predict(f["valid_hist"].loc[~f["known"],MASKED_BASE+BASE_HIST]),~f["known"],"masked","11"))
KNOWN_OOF=pd.concat(known_parts,ignore_index=True)
MB=pd.concat(masked_base,ignore_index=True); MH=pd.concat(masked_hist,ignore_index=True)
M11=MB.copy(); M11["prediction"]=.65*MB.prediction.to_numpy()+.35*MH.prediction.to_numpy()
BASE_OOF=pd.concat([KNOWN_OOF,M11],ignore_index=True)
BASELINE=add("Notebook 11 public-best",BASE_OOF,"baseline")
display(pd.DataFrame([BASELINE["metrics"]]))
print("Fold RMSE",BASE_OOF.groupby("fold").apply(lambda z:root_mean_squared_error(z.target,z.prediction),include_groups=False).to_dict())
print("OOF predictions",len(BASE_OOF),BASE_OOF.sample_id.nunique())


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows
0,Notebook 11 public-best,0.692625,0.586791,0.739881,0.522616,0.769866,0.056282,0.703452,0.744714,0.805396,0.726006,842834


Fold RMSE {1: 0.6528958055814488, 2: 0.648190538634666, 3: 0.7698659668926635}
OOF predictions 842834 842834


## 4. Persistence, climatology ve recent-trend baselines

In [5]:
def stat_predictions(f):
    tr=f["train"]; va=f["valid"]; glob=tr.target.mean()
    gm=tr.groupby("month").target.mean(); latm=tr.groupby(["lat_band","month"]).target.mean()
    cell=add_spatial(tr,10).groupby(["cell_10_lat","cell_10_lon","month"]).target.mean()
    loc=tr.groupby(["lat","lon"]).target.agg(["mean","median"])
    lm=tr.groupby(["lat","lon","month"]).target.mean()
    av=add_spatial(va,10)
    out={}
    out["Global monthly climatology"]=va.month.map(gm).fillna(glob).to_numpy()
    out["Latitude-band monthly climatology"]=pd.Series(list(zip(va.lat_band,va.month))).map(latm).fillna(glob).to_numpy()
    out["Spatial-cell monthly climatology"]=pd.Series(list(zip(av.cell_10_lat,av.cell_10_lon,av.month))).map(cell).fillna(glob).to_numpy()
    out["Location expanding mean"]=pd.Series(list(zip(va.lat,va.lon))).map(loc["mean"]).fillna(glob).to_numpy()
    out["Location expanding median"]=pd.Series(list(zip(va.lat,va.lon))).map(loc["median"]).fillna(glob).to_numpy()
    out["Location monthly expanding mean"]=pd.Series(list(zip(va.lat,va.lon,va.month))).map(lm).fillna(pd.Series(list(zip(va.lat,va.lon))).map(loc["mean"])).fillna(glob).to_numpy()
    # Last two training observations only; extrapolation horizon capped to avoid explosive drift.
    tail=tr.sort_values("time").groupby(["lat","lon"]).tail(2)
    last=tail.groupby(["lat","lon"]).tail(1).set_index(["lat","lon"])
    slope=tail.groupby(["lat","lon"]).target.agg(lambda z:z.iloc[-1]-z.iloc[-2] if len(z)>1 else 0)
    keys=pd.MultiIndex.from_frame(va[["lat","lon"]]); last_y=last.target.reindex(keys).to_numpy(); sl=slope.reindex(keys).fillna(0).clip(-.5,.5).to_numpy()
    last_t=pd.DatetimeIndex(last.time.reindex(keys)); gap=((va.time.dt.year.to_numpy()-last_t.year)*12+va.time.dt.month.to_numpy()-last_t.month)
    out["Recent safe trend"]=np.where(np.isfinite(last_y),last_y+sl*np.minimum(gap,3),glob)
    return out
stats={}
for f in FOLDS:
    for n,p in stat_predictions(f).items():
        z1=frame(f,p[f["known"]],f["known"],"known",n)
        z2=frame(f,p[~f["known"]],~f["known"],"masked",n)
        stats.setdefault(n,[]).append(pd.concat([z1,z2],ignore_index=True))
    mask=f["known"]; p=f["valid"].loc[mask,"TWS_t"].to_numpy()
    stats.setdefault("Persistence known",[]).append(frame(f,p,mask,"known","Persistence known"))
for n,parts in stats.items():
    o=pd.concat(parts,ignore_index=True)
    if n=="Persistence known":
        print(n,root_mean_squared_error(o.target,o.prediction),len(o))
    else:add(n,o,"baseline")
display(pd.DataFrame([e["metrics"] for e in EXPS if e["kind"]=="baseline"]))


Persistence known 0.6269418025152858 280924


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows
0,Notebook 11 public-best,0.692625,0.586791,0.739881,0.522616,0.769866,0.056282,0.703452,0.744714,0.805396,0.726006,842834
1,Global monthly climatology,0.870514,0.863752,0.873874,0.680051,0.886710,0.011607,1.030640,0.925406,0.889719,0.877465,842834
2,Latitude-band monthly climatology,0.933909,0.921917,0.939847,0.739590,0.970442,0.032129,1.076242,1.035575,0.967678,0.951075,842834
3,Spatial-cell monthly climatology,1.014318,0.996559,1.023081,0.822781,1.043814,0.028965,1.132444,1.121340,1.078924,1.028933,842834
4,Location expanding mean,1.029531,1.006189,1.041004,0.845412,1.077616,0.040139,1.144952,1.136350,1.114752,1.051587,842834
5,Location expanding median,1.047860,1.024951,1.059128,0.858740,1.100056,0.046649,1.161271,1.153119,1.136814,1.072571,842834
6,Location monthly expanding mean,1.055104,1.035137,1.064946,0.866970,1.107555,0.043516,1.166452,1.163826,1.143996,1.079096,842834
7,Recent safe trend,1.581350,1.526450,1.608094,1.231340,1.650702,0.050556,1.549788,1.792783,1.690412,1.611327,842834


## 5. Known-TWS expert: absolute, delta, normalized delta ve recursive

In [6]:
DELTA_FEATURES=KNOWN_FEATURES+["hist_delta_mean","hist_delta_std","hist_delta_median","hist_delta_roll3",
    "hist_delta_roll6","hist_delta_roll12","hist_trend","months_since"]
known_candidates={"Known absolute LGBM":KNOWN_OOF.copy()}
for mode in ["delta","normalized_delta"]:
    parts=[]
    for f in FOLDS:
        tr=f["train_hist"]; va=f["valid_hist"]; use=tr.TWS_t.notna()
        delta=tr.loc[use,"target"]-tr.loc[use,"TWS_t"]
        if mode=="normalized_delta":
            scale=delta.groupby([tr.loc[use,"lat"],tr.loc[use,"lon"]]).std().clip(lower=.05)
            keys=pd.MultiIndex.from_frame(tr.loc[use,["lat","lon"]]); sc=scale.reindex(keys).fillna(delta.std()).to_numpy()
            y=delta.to_numpy()/sc
        else:y=delta
        m=lgb().fit(tr.loc[use,DELTA_FEATURES],y)
        v=va.loc[f["known"]]; q=m.predict(v[DELTA_FEATURES])
        if mode=="normalized_delta":
            vk=pd.MultiIndex.from_frame(v[["lat","lon"]]); q=q*scale.reindex(vk).fillna(delta.std()).to_numpy()
        p=v.TWS_t.to_numpy()+q
        parts.append(frame(f,p,f["known"],"known",mode))
    known_candidates[mode]=pd.concat(parts,ignore_index=True)
    add("Known "+mode,known_candidates[mode],"known")
# OOF alpha fixed grid; report sensitivity, choose mean across fold-optimal alphas.
alpha_rows=[]
for a in np.arange(0,1.01,.1):
    p=(1-a)*known_candidates["Known absolute LGBM"].prediction.to_numpy()+a*known_candidates["delta"].prediction.to_numpy()
    z=known_candidates["delta"].copy(); z["prediction"]=p
    fs=z.groupby("fold").apply(lambda q:root_mean_squared_error(q.target,q.prediction),include_groups=False)
    alpha_rows.append((a,root_mean_squared_error(z.target,z.prediction),fs.max(),fs.std(ddof=0)))
alpha_table=pd.DataFrame(alpha_rows,columns=["alpha","rmse","worst","std"])
KNOWN_ALPHA=float(alpha_table.sort_values(["rmse","worst"]).iloc[0].alpha)
blend=known_candidates["delta"].copy(); blend["prediction"]=(1-KNOWN_ALPHA)*known_candidates["Known absolute LGBM"].prediction.to_numpy()+KNOWN_ALPHA*blend.prediction.to_numpy()
known_candidates["Known conservative blend"]=blend
add("Known conservative blend",blend,"known",{"alpha":KNOWN_ALPHA})
# Recursive: starts from last training target, then carries only its own predicted delta.
rec=[]
for f,part in zip(FOLDS,[known_candidates["delta"][known_candidates["delta"].fold==i] for i in [1,2,3]]):
    v=f["valid"].copy(); pred_map=dict(zip(part.sample_id,part.prediction-part.TWS_t))
    state=f["train"].sort_values("time").groupby(["lat","lon"]).tail(1).set_index(["lat","lon"]).target.to_dict(); pp=[]
    for _,r in v.sort_values(["time","lat","lon"]).iterrows():
        key=(r.lat,r.lon); d=pred_map.get(r.sample_id,0.0); val=state.get(key,np.nan)+d if key in state else np.nan
        pp.append((r.sample_id,val)); state[key]=val
    mp=dict(pp); mask=f["known"]; q=f["valid"].loc[mask,"sample_id"].map(mp).fillna(f["valid"].loc[mask,"TWS_t"]).to_numpy()
    rec.append(frame(f,q,mask,"known","Recursive delta"))
recursive=pd.concat(rec,ignore_index=True); add("Known recursive delta",recursive,"known")
display(alpha_table)
display(pd.DataFrame([e["metrics"] for e in EXPS if e["kind"]=="known"]))


,alpha,rmse,worst,std
0,0.0,0.586791,0.693215,0.080267
1,0.1,0.584045,0.691531,0.081056
2,0.2,0.581851,0.690307,0.081767
3,0.3,0.580216,0.689546,0.082396
4,0.4,0.579144,0.689250,0.082941
5,0.5,0.578637,0.689418,0.083400
6,0.6,0.578699,0.690052,0.083771
7,0.7,0.579327,0.691148,0.084055
8,0.8,0.580522,0.692706,0.084253
9,0.9,0.582278,0.694722,0.084364


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows
0,Known delta,0.584592,0.584592,NaN,0.419503,0.697192,0.084392,0.581534,0.620228,0.728340,0.633840,280924
1,Known normalized_delta,0.587239,0.587239,NaN,0.415978,0.698815,0.083086,0.573642,0.626234,0.732677,0.635904,280924
2,Known conservative blend,0.578637,0.578637,NaN,0.415774,0.689418,0.083400,0.578706,0.618721,0.721036,0.627182,280924
3,Known recursive delta,1.175584,1.175584,NaN,0.851799,1.332010,0.116114,1.222484,1.371544,1.417785,1.243719,280924


## 6. Masked-TWS expert, spatial çözünürlük ve residual modelleme

Masked modellerde `TWS_t` feature değildir; `tws_missing` açıkça bulunur. Ağır aileler önce fold training'in deterministik alt örneğinde pilot edilir. Pilot Extra Trees `14`te baseline'ı geçmediği için burada full-data tekrarına alınmaz; CatBoost yalnız kuruluysa kontrollü 250 iterasyonla değerlendirilir.


In [7]:
masked_candidates={"Masked absolute LGBM":MB.copy(),"Masked 11 blend":M11.copy()}
def masked_lgb_experiment(name,extra_builder=None,features=MASKED_FEATURES):
    parts=[]
    for f in FOLDS:
        tr,va=(f["train"],f["valid"]) if extra_builder is None else extra_builder(f)
        m=lgb().fit(tr[features],tr.target); mask=~f["known"]
        parts.append(frame(f,m.predict(va.loc[mask,features]),mask,"masked",name))
    z=pd.concat(parts,ignore_index=True); masked_candidates[name]=z; return z
for res in [20,10,5]:
    feat=MASKED_FEATURES+[f"cell_{res}_lat",f"cell_{res}_lon",f"cell{res}_month_mean"]
    def builder(f,res=res):
        return causal_spatial_train(f["train"],res),apply_spatial(f["train"],f["valid"],res)
    z=masked_lgb_experiment(f"Masked spatial {res}deg",builder,feat)
    add(f"Masked spatial {res}deg",z,"masked",{"resolution":res})
# Climatology + ML residual (causal cell-month base).
res_parts=[]
for f in FOLDS:
    tr=causal_spatial_train(f["train"],10); va=apply_spatial(f["train"],f["valid"],10); c="cell10_month_mean"
    fallback=tr.target.expanding().mean().shift(1); base_tr=tr[c].fillna(fallback).fillna(tr.target.mean())
    y=tr.target-base_tr
    feat=MASKED_FEATURES+[c]; m=lgb().fit(tr[feat],y)
    mask=~f["known"]; base_va=va.loc[mask,c].fillna(f["train"].target.mean())
    p=base_va.to_numpy()+m.predict(va.loc[mask,feat])
    res_parts.append(frame(f,p,mask,"masked","Spatial climatology + ML residual"))
spres=pd.concat(res_parts,ignore_index=True); masked_candidates["Spatial climatology + ML residual"]=spres
add("Spatial climatology + ML residual",spres,"masked")
# Regime feature and separate experts.
reg_parts=[]; expert_parts=[]
for f in FOLDS:
    tr=f["train"].copy(); va=f["valid"].copy()
    qs=tr.SPEI_03_t.quantile([.1,.3,.7,.9]).to_numpy()
    def regime(x): return np.digitize(x,qs).astype("int8")
    tr["regime"]=regime(tr.SPEI_03_t); va["regime"]=regime(va.SPEI_03_t); feat=MASKED_FEATURES+["regime"]
    global_m=lgb().fit(tr[feat],tr.target); mask=~f["known"]
    reg_parts.append(frame(f,global_m.predict(va.loc[mask,feat]),mask,"masked","Global + regime"))
    p=np.empty(mask.sum()); vv=va.loc[mask]
    for r in range(5):
        a=tr[tr.regime==r]
        if len(a)<10000: p[vv.regime.to_numpy()==r]=global_m.predict(vv.loc[vv.regime==r,feat])
        else:
            mm=lgb().fit(a[feat],a.target); p[vv.regime.to_numpy()==r]=mm.predict(vv.loc[vv.regime==r,feat])
    expert_parts.append(frame(f,p,mask,"masked","Separate regime experts"))
for n,parts in [("Global + regime",reg_parts),("Separate regime experts",expert_parts)]:
    z=pd.concat(parts,ignore_index=True); masked_candidates[n]=z; add(n,z,"masked")
display(pd.DataFrame([e["metrics"] for e in EXPS if e["kind"]=="masked"]).sort_values("Robust score"))


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows
3,Spatial climatology + ML residual,0.744878,NaN,0.744878,0.574474,0.803602,0.042634,0.754631,0.800977,0.831773,0.770217,561910
2,Masked spatial 5deg,0.746123,NaN,0.746123,0.575829,0.804695,0.042626,0.764298,0.792840,0.835827,0.771422,561910
1,Masked spatial 10deg,0.748440,NaN,0.748440,0.576909,0.808469,0.044049,0.767595,0.797920,0.839059,0.774459,561910
4,Global + regime,0.743793,NaN,0.743793,0.576136,0.815238,0.051814,0.747438,0.800987,0.845900,0.774607,561910
0,Masked spatial 20deg,0.748219,NaN,0.748219,0.577385,0.810086,0.044953,0.760621,0.796724,0.840031,0.774924,561910
5,Separate regime experts,0.748312,NaN,0.748312,0.579503,0.815231,0.048657,0.744657,0.830352,0.844649,0.777206,561910


## 7. Dual experts, stress tests ve final seçim kuralları

In [8]:
def dual(name,k,m):
    z=pd.concat([known_candidates[k],masked_candidates[m]],ignore_index=True); return add(name,z,"dual",{"known":k,"masked":m})
dual_models=[]
for k in ["Known absolute LGBM","delta","normalized_delta","Known conservative blend"]:
    for m in ["Masked 11 blend","Masked spatial 20deg","Masked spatial 10deg","Masked spatial 5deg","Spatial climatology + ML residual","Global + regime","Separate regime experts"]:
        dual_models.append(dual(f"Dual | {k} | {m}",k,m))
comparison=pd.DataFrame([e["metrics"] for e in [BASELINE,*dual_models]])
b=BASELINE["metrics"]
comparison["Overall gain"]=b["Overall RMSE"]-comparison["Overall RMSE"]
comparison["Worst gain"]=b["Worst-fold RMSE"]-comparison["Worst-fold RMSE"]
comparison["Std gain"]=b["Fold std"]-comparison["Fold std"]
comparison["Masked gain"]=b["Masked RMSE"]-comparison["Masked RMSE"]
comparison["Known degradation"]=comparison["Known RMSE"]-b["Known RMSE"]
def passes(r):
    return ((r["Overall gain"]>=MIN_OVERALL_PRIMARY and r["Worst gain"]>=0) or
            (r["Overall gain"]>=ALT_OVERALL and r["Worst gain"]>=ALT_WORST and r["Std gain"]>0) or
            (abs(r["Overall gain"])<=.0008 and r["Masked gain"]>=MASKED_GAIN and r["Known degradation"]<=.0005))
comparison["Passes fixed rule"]=comparison.apply(passes,axis=1)
eligible=comparison[comparison["Passes fixed rule"]].sort_values(["Robust score","Overall RMSE"])
PRIMARY_NAME=eligible.iloc[0].Model if len(eligible) else "Notebook 11 public-best"
PRIMARY=next(e for e in [BASELINE,*dual_models] if e["name"]==PRIMARY_NAME)
display(comparison.sort_values("Robust score").head(20))
print("PRIMARY",PRIMARY_NAME)
# Stress-test slices already in table; latitude-region holdout proxy reports each band without refitting.
stress=PRIMARY["oof"].assign(lat_region=pd.cut(PRIMARY["oof"].lat,[-90,-30,0,30,90]))
display(stress.groupby("lat_region",observed=True).apply(lambda z:pd.Series({"rows":len(z),"rmse":root_mean_squared_error(z.target,z.prediction)}),include_groups=False))


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows,Overall gain,Worst gain,Std gain,Masked gain,Known degradation,Passes fixed rule
22,Dual | Known conservative blend | Masked 11 blend,0.690335,0.578637,0.739881,0.520002,0.768729,0.057137,0.700509,0.739912,0.803866,0.724217,842834,0.002290,0.001137,-0.000855,0.000000,-0.008153,True
26,Dual | Known conservative blend | Spatial clim...,0.693908,0.578637,0.744878,0.521578,0.767450,0.053474,0.700369,0.745582,0.801174,0.725662,842834,-0.001283,0.002416,0.002808,-0.004997,-0.008153,False
0,Notebook 11 public-best,0.692625,0.586791,0.739881,0.522616,0.769866,0.056282,0.703452,0.744714,0.805396,0.726006,842834,0.000000,0.000000,0.000000,0.000000,0.000000,False
1,Dual | Known absolute LGBM | Masked 11 blend,0.692625,0.586791,0.739881,0.522616,0.769866,0.056282,0.703452,0.744714,0.805396,0.726006,842834,0.000000,0.000000,0.000000,0.000000,0.000000,False
8,Dual | delta | Masked 11 blend,0.692005,0.584592,0.739881,0.521245,0.771062,0.057616,0.701297,0.740330,0.805452,0.726173,842834,0.000620,-0.001196,-0.001334,0.000000,-0.002199,False
25,Dual | Known conservative blend | Masked spati...,0.694799,0.578637,0.746123,0.522481,0.768214,0.053385,0.707289,0.739741,0.803170,0.726499,842834,-0.002174,0.001652,0.002898,-0.006241,-0.008153,False
15,Dual | normalized_delta | Masked 11 blend,0.692751,0.587239,0.739881,0.520070,0.771551,0.057433,0.699106,0.742002,0.808295,0.726809,842834,-0.000127,-0.001685,-0.001150,0.000000,0.000448,False
5,Dual | Known absolute LGBM | Spatial climatolo...,0.696186,0.586791,0.744878,0.524192,0.768589,0.052616,0.703312,0.750348,0.802709,0.727441,842834,-0.003561,0.001277,0.003666,-0.004997,0.000000,False
12,Dual | delta | Spatial climatology + ML residual,0.695569,0.584592,0.744878,0.522821,0.769786,0.053984,0.701157,0.745996,0.802765,0.727620,842834,-0.002944,0.000080,0.002298,-0.004997,-0.002199,False
19,Dual | normalized_delta | Spatial climatology ...,0.696312,0.587239,0.744878,0.521646,0.770277,0.053866,0.698965,0.747656,0.805617,0.728270,842834,-0.003687,-0.000411,0.002416,-0.004997,0.000448,False


PRIMARY Dual | Known conservative blend | Masked 11 blend


,rows,rmse
lat_region,,
"(-90, -30]",30741.0,0.750737
"(-30, 0]",135244.0,0.753252
"(0, 30]",166204.0,0.697573
"(30, 90]",510645.0,0.666331


## 8. Ensemble diversity ve fold-stable simplex blend

In [9]:
# Candidate OOF matrix: baseline, best dual, spatial dual and 14-style tree OOF if available.
cands=[BASELINE]
if PRIMARY["name"]!=BASELINE["name"]: cands.append(PRIMARY)
for e in sorted(dual_models,key=lambda x:x["metrics"]["Robust score"]):
    if e["name"] not in {x["name"] for x in cands}:
        ref=BASELINE["oof"][["sample_id","target"]]
        a=ref.merge(e["oof"][["sample_id","prediction"]],on="sample_id").prediction
        keep=True
        for x in cands:
            xp=ref.merge(x["oof"][["sample_id","prediction"]],on="sample_id").prediction
            if np.corrcoef(a,xp)[0,1]>CORR_LIMIT: keep=False
        if keep:cands.append(e)
    if len(cands)>=4:break
if len(cands)==1: cands.append(sorted(dual_models,key=lambda x:x["metrics"]["Robust score"])[0])
ref=BASELINE["oof"][["sample_id","target","fold","segment","time","lat","lon",*SPEI,"SOIL_MOISTURE_t","TWS_t"]].copy()
P=[]
for e in cands:P.append(ref[["sample_id"]].merge(e["oof"][["sample_id","prediction"]],on="sample_id").prediction.to_numpy())
P=np.column_stack(P); names=[e["name"] for e in cands]
rescorr=pd.DataFrame(P-ref.target.to_numpy()[:,None],columns=names).corr()
display(rescorr)
grid=[]
if P.shape[1]>=2:
    for w in np.arange(0,1.01,.1):
        pred=w*P[:,0]+(1-w)*P[:,1]
        fs=[root_mean_squared_error(ref.loc[ref.fold==f,"target"],pred[ref.fold==f]) for f in [1,2,3]]
        grid.append((w,root_mean_squared_error(ref.target,pred),max(fs),np.std(fs)))
blend_grid=pd.DataFrame(grid,columns=["baseline_weight","rmse","worst","std"])
fold_weights=[]
for f in [1,2,3]:
    q=blend_grid.copy()
    # Same global grid is diagnostic; fold optimum calculated directly.
    vals=[]
    ix=ref.fold==f
    for w in q.baseline_weight: vals.append(root_mean_squared_error(ref.loc[ix,"target"],w*P[ix,0]+(1-w)*P[ix,1]))
    fold_weights.append(q.iloc[int(np.argmin(vals))].baseline_weight)
WEIGHT_STABLE=(max(fold_weights)-min(fold_weights)<=.3)
best_blend=blend_grid.sort_values(["rmse","worst"]).iloc[0]
blend_pred=best_blend.baseline_weight*P[:,0]+(1-best_blend.baseline_weight)*P[:,1]
blend_oof=ref.copy(); blend_oof["prediction"]=blend_pred
BLEND=add("Stable simplex blend",blend_oof,"ensemble",{"weight":float(best_blend.baseline_weight),"stable":WEIGHT_STABLE})
median_oof=ref.copy(); median_oof["prediction"]=np.median(P,axis=1)
MEDIAN=add("Diverse median blend",median_oof,"ensemble")
print("Fold-optimal baseline weights",fold_weights,"stable",WEIGHT_STABLE)
display(pd.DataFrame([BLEND["metrics"],MEDIAN["metrics"]]))


,Notebook 11 public-best,Dual | Known conservative blend | Masked 11 blend,Dual | Known conservative blend | Spatial climatology + ML residual,Dual | Known conservative blend | Masked spatial 5deg
Notebook 11 public-best,1.000000,0.997152,0.982160,0.983750
Dual | Known conservative blend | Masked 11 blend,0.997152,1.000000,0.984946,0.986504
Dual | Known conservative blend | Spatial climatology + ML residual,0.982160,0.984946,1.000000,0.986028
Dual | Known conservative blend | Masked spatial 5deg,0.983750,0.986504,0.986028,1.000000


Fold-optimal baseline weights [np.float64(0.0), np.float64(0.0), np.float64(0.2)] stable True


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows
0,Stable simplex blend,0.690335,0.578637,0.739881,0.520002,0.768729,0.057137,0.700509,0.739912,0.803866,0.724217,842834
1,Diverse median blend,0.689242,0.578637,0.738352,0.518711,0.766519,0.056289,0.698742,0.738012,0.801330,0.722633,842834


## 9. Full fit, test prediction ve en fazla üç anlamlı submission

In [10]:
def fit_full_components():
    known_abs=lgb().fit(train_f[KNOWN_FEATURES],train_f.target)
    train_h=causal_history(train_f); test_h=history_state_apply(train_f,test_f)
    use=train_h.TWS_t.notna(); delta=train_h.loc[use,"target"]-train_h.loc[use,"TWS_t"]
    kd=lgb().fit(train_h.loc[use,DELTA_FEATURES],delta)
    known_pred_abs=known_abs.predict(test_f[KNOWN_FEATURES])
    known_pred_delta=test_f.TWS_t.to_numpy()+kd.predict(test_h[DELTA_FEATURES])
    mb=lgb().fit(train_f[MASKED_BASE],train_f.target)
    mh=lgb().fit(train_h[MASKED_BASE+BASE_HIST],train_h.target)
    masked11=.65*mb.predict(test_f[MASKED_BASE])+.35*mh.predict(test_h[MASKED_BASE+BASE_HIST])
    return train_h,test_h,known_pred_abs,known_pred_delta,masked11
train_h,test_h,kabs,kdelta,m11test=fit_full_components()
known_test=test_f.TWS_t.notna().to_numpy()
baseline_test=np.where(known_test,kabs,m11test)
def full_dual_prediction(e):
    k=e["meta"]["known"]; m=e["meta"]["masked"]
    if k=="Known absolute LGBM": kp=kabs
    elif k=="delta": kp=kdelta
    elif k=="Known conservative blend": kp=(1-KNOWN_ALPHA)*kabs+KNOWN_ALPHA*kdelta
    elif k=="normalized_delta":
        use=train_h.TWS_t.notna(); d=train_h.loc[use,"target"]-train_h.loc[use,"TWS_t"]
        scale=d.groupby([train_h.loc[use,"lat"],train_h.loc[use,"lon"]]).std().clip(lower=.05)
        keys=pd.MultiIndex.from_frame(train_h.loc[use,["lat","lon"]]); sc=scale.reindex(keys).fillna(d.std()).to_numpy()
        nm=lgb().fit(train_h.loc[use,DELTA_FEATURES],d.to_numpy()/sc)
        tk=pd.MultiIndex.from_frame(test_h[["lat","lon"]]); kp=test_h.TWS_t.to_numpy()+nm.predict(test_h[DELTA_FEATURES])*scale.reindex(tk).fillna(d.std()).to_numpy()
    if m=="Masked 11 blend": mp=m11test
    elif m.startswith("Masked spatial"):
        res=int(m.split()[-1].replace("deg","")); tr=causal_spatial_train(train_f,res); te=apply_spatial(train_f,test_f,res)
        feat=MASKED_FEATURES+[f"cell_{res}_lat",f"cell_{res}_lon",f"cell{res}_month_mean"]
        mp=lgb().fit(tr[feat],tr.target).predict(te[feat])
    elif m=="Spatial climatology + ML residual":
        tr=causal_spatial_train(train_f,10); te=apply_spatial(train_f,test_f,10); c="cell10_month_mean"
        base=tr[c].fillna(tr.target.expanding().mean().shift(1)).fillna(tr.target.mean()); feat=MASKED_FEATURES+[c]
        mm=lgb().fit(tr[feat],tr.target-base); mp=te[c].fillna(train_f.target.mean()).to_numpy()+mm.predict(te[feat])
    else:
        tr=train_f.copy(); te=test_f.copy(); qs=tr.SPEI_03_t.quantile([.1,.3,.7,.9]).to_numpy()
        tr["regime"]=np.digitize(tr.SPEI_03_t,qs).astype("int8"); te["regime"]=np.digitize(te.SPEI_03_t,qs).astype("int8"); feat=MASKED_FEATURES+["regime"]
        if m=="Global + regime": mp=lgb().fit(tr[feat],tr.target).predict(te[feat])
        else:
            gm=lgb().fit(tr[feat],tr.target); mp=gm.predict(te[feat])
            for r in range(5):
                a=tr[tr.regime==r]; ix=te.regime==r
                if len(a)>=10000: mp[ix]=lgb().fit(a[feat],a.target).predict(te.loc[ix,feat])
    return np.where(known_test,kp,mp)
primary_test=baseline_test if PRIMARY is BASELINE else full_dual_prediction(PRIMARY)
test_predictions={"Notebook 11":baseline_test,"Primary":primary_test}
component_test=[]
for e in cands:
    if e is BASELINE: component_test.append(baseline_test)
    elif e is PRIMARY: component_test.append(primary_test)
    else: component_test.append(full_dual_prediction(e))
diverse_test=np.median(np.column_stack(component_test),axis=1)
test_predictions["Diverse"]=diverse_test
if WEIGHT_STABLE and BLEND["metrics"]["Robust score"]<BASELINE["metrics"]["Robust score"]:
    w=BLEND["meta"]["weight"]; test_predictions["Conservative"]=w*baseline_test+(1-w)*primary_test


In [11]:
def validate_write(label,pred,path):
    byid=pd.Series(np.asarray(pred,float),index=test_f.ID.astype(str)); sub=sample.copy()
    sub["Target"]=sample.ID.astype(str).map(byid).to_numpy(float)
    checks={"rows_280961":len(sub)==280_961,"columns":list(sub.columns)==["ID","Target"],
      "id_set":set(sub.ID.astype(str))==set(test_f.ID.astype(str)),"unique":sub.ID.is_unique,
      "no_nan":sub.Target.notna().all(),"finite":np.isfinite(sub.Target).all(),
      "numeric":pd.api.types.is_numeric_dtype(sub.Target),"order":sub.ID.equals(sample.ID),
      "known_rows":int(known_test.sum()),"masked_rows":int((~known_test).sum())}
    assert all(v for k,v in checks.items() if isinstance(v,(bool,np.bool_))),checks
    sub.to_csv(path,index=False); reread=pd.read_csv(path)
    assert len(reread)==280_961 and reread.ID.equals(sample.ID) and np.isfinite(reread.Target).all()
    return {"Candidate":label,**checks,"mean":sub.Target.mean(),"std":sub.Target.std(),"min":sub.Target.min(),"max":sub.Target.max(),"path":str(path)}
files=[]; candidate_rows=[]
if PRIMARY_NAME!="Notebook 11 public-best":
    files.append(validate_write("Primary",primary_test,OUT/"submission_15_primary.csv"))
    candidate_rows.append({"Candidate":"Primary",**PRIMARY["metrics"],"Prediction mean":primary_test.mean(),"Prediction std":primary_test.std(),
      "Minimum":primary_test.min(),"Maximum":primary_test.max(),"Corr vs 11":np.corrcoef(primary_test,baseline_test)[0,1]})
diverse_corr_primary=np.corrcoef(diverse_test,primary_test)[0,1]
diverse_corr_base=np.corrcoef(diverse_test,baseline_test)[0,1]
if (MEDIAN["metrics"]["Robust score"]<BASELINE["metrics"]["Robust score"] and
    diverse_corr_primary<=CORR_LIMIT and diverse_corr_base<=CORR_LIMIT):
    files.append(validate_write("Diverse",diverse_test,OUT/"submission_15_diverse.csv"))
    candidate_rows.append({"Candidate":"Diverse",**MEDIAN["metrics"],"Prediction mean":diverse_test.mean(),
      "Prediction std":diverse_test.std(),"Minimum":diverse_test.min(),"Maximum":diverse_test.max(),
      "Corr vs 11":diverse_corr_base})
if "Conservative" in test_predictions:
    cp=test_predictions["Conservative"]; corr=np.corrcoef(cp,primary_test)[0,1]
    if corr<=CORR_LIMIT:
        files.append(validate_write("Conservative",cp,OUT/"submission_15_conservative.csv"))
        candidate_rows.append({"Candidate":"Conservative",**BLEND["metrics"],"Prediction mean":cp.mean(),"Prediction std":cp.std(),
          "Minimum":cp.min(),"Maximum":cp.max(),"Corr vs 11":np.corrcoef(cp,baseline_test)[0,1]})
display(pd.DataFrame(candidate_rows))
display(pd.DataFrame(files))
if len(test_predictions)>1:
    arr=pd.DataFrame(test_predictions)
    display(arr.corr())
    print("Mean absolute differences\n",arr.apply(lambda a:arr.apply(lambda b:np.mean(np.abs(a-b)))))
print("Baseline test range",baseline_test.min(),baseline_test.max())


,Candidate,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows,Prediction mean,Prediction std,Minimum,Maximum,Corr vs 11
0,Primary,Dual | Known conservative blend | Masked 11 blend,0.690335,0.578637,0.739881,0.520002,0.768729,0.057137,0.700509,0.739912,0.803866,0.724217,842834,-0.137739,0.543589,-2.3482,2.206941,0.990041


,Candidate,rows_280961,columns,id_set,unique,no_nan,finite,numeric,order,known_rows,masked_rows,mean,std,min,max,path
0,Primary,True,True,True,True,True,True,True,True,94048,186913,-0.137739,0.54359,-2.3482,2.206941,C:\global-water-storage-forecasting\submission...


,Notebook 11,Primary,Diverse,Conservative
Notebook 11,1.000000,0.990041,0.987172,0.990041
Primary,0.990041,1.000000,0.997281,1.000000
Diverse,0.987172,0.997281,1.000000,0.997281
Conservative,0.990041,1.000000,0.997281,1.000000


Mean absolute differences
               Notebook 11   Primary   Diverse  Conservative
Notebook 11      0.000000  0.031529  0.051395      0.031529
Primary          0.031529  0.000000  0.019865      0.000000
Diverse          0.051395  0.019865  0.000000      0.019865
Conservative     0.031529  0.000000  0.019865      0.000000
Baseline test range -2.2557162590905557 2.0545858417354426


## 10. Büyük karşılaştırma ve final rapor

In [12]:
all_table=pd.DataFrame([e["metrics"] for e in EXPS]).sort_values(["Robust score","Overall RMSE"])
display(all_table)
def answer_report():
    best=all_table.iloc[0]; robust=all_table.sort_values("Robust score").iloc[0]
    abs_rmse=summarize(known_candidates["Known absolute LGBM"],"x")["Known RMSE"]
    delta_rmse=summarize(known_candidates["delta"],"x")["Known RMSE"]
    pers=root_mean_squared_error(pd.concat(stats["Persistence known"]).target,pd.concat(stats["Persistence known"]).prediction)
    spatial=all_table[all_table.Model.str.contains("spatial",case=False)]
    residual=all_table[all_table.Model.str.contains("residual",case=False)]
    regime=all_table[all_table.Model.str.contains("regime",case=False)]
    recursive_rmse=summarize(recursive,"x")["Known RMSE"]
    return Markdown(f'''
### Final rapor

1. **Absolute mı delta mı?** Known absolute RMSE `{abs_rmse:.6f}`, delta RMSE `{delta_rmse:.6f}`; düşük olan daha iyidir. Normalized delta ve OOF alpha tablolarda ayrıca gösterildi.
2. **Persistence:** Known satırlarda RMSE `{pers:.6f}`; gerçek bir zaman-serisi referansıdır.
3. **Climatology:** Global, latitude-band, spatial-cell, location mean/median/monthly sonuçları ayrı hesaplandı; baseline tablosunda doğrudan karşılaştırılabilir.
4. **Known ve masked farkı:** Notebook 11 known `{b["Known RMSE"]:.6f}`, masked `{b["Masked RMSE"]:.6f}`; masked problem belirgin biçimde daha zordur.
5. **Spatial katkı:** En iyi spatial deney `{spatial.iloc[0].Model if len(spatial) else "yok"}` / `{spatial.iloc[0]["Overall RMSE"] if len(spatial) else np.nan:.6f}`.
6. **Residual modelling:** En iyi residual deney `{residual.iloc[0].Model if len(residual) else "yok"}` / `{residual.iloc[0]["Overall RMSE"] if len(residual) else np.nan:.6f}`.
7. **Regime experts:** En iyi regime deney `{regime.iloc[0].Model if len(regime) else "yok"}` / `{regime.iloc[0]["Overall RMSE"] if len(regime) else np.nan:.6f}`; yalnız sabit seçim kuralını geçerse kullanıldı.
8. **Recursive forecasting:** Known RMSE `{recursive_rmse:.6f}`. Teacher forcing yoktur; kendi tahmini ileri taşındı. Baseline'dan kötü ise güvenilir sayılmaz.
9. **En düşük ortalama RMSE:** `{best.Model}` / `{best["Overall RMSE"]:.6f}`.
10. **En iyi robust score:** `{robust.Model}` / `{robust["Robust score"]:.6f}`.
11. **Public genelleme adayı:** `{PRIMARY_NAME}`; yalnız önceden sabit overall/worst/std/masked kurallarından geçtiyse primary seçildi.
12. **Private için iki aday:** Üretilen dosyalar `{[x["Candidate"] for x in files]}`. Korelasyon `>{CORR_LIMIT}` ise gereksiz ikinci dosya yazılmadı.
13. **Notebook 11 neden güçlü?** Known/masked routing, missing-aware nonlinear model ve causal climatology blend'i farklı rejimlerde düşük varyans sağlıyor. Yeni model sabit kuralı geçmediyse 11 korunur.
14. **Local→public beklentisi:** Seçim yalnız ortalama OOF'a değil worst-fold, fold std, recent/dry/wet streslerine ve sabit robust score'a bağlıdır. Bu garanti değildir; public skor tuning amacıyla kullanılmadı.

**Pilot/ölçek notu:** KNN/GPR/yoğun robust modeller 2.15M satırda uygulanabilir değildir. `14`te full-validation pilotu LightGBM'i geçmeyen Extra Trees/Random Forest tekrar full-data ana aday yapılmadı. CatBoost kurululuğu raporlandı; geniş arama yapılmadı. Spatial bellek etkisi çözünürlükle oluşan hücre sayısı üzerinden aşağıda raporlanır.
''')
display(answer_report())
for res in [20,10,5]:
    x=add_spatial(train_f,res); print(res,"degrees cells",x[[f"cell_{res}_lat",f"cell_{res}_lon"]].drop_duplicates().shape[0],
      "feature memory MB",x[[f"cell_{res}_lat",f"cell_{res}_lon"]].memory_usage(deep=True).sum()/1024**2)


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std,Dry RMSE,Wet RMSE,Recent RMSE,Robust score,OOF rows
10,Known conservative blend,0.578637,0.578637,NaN,0.415774,0.689418,0.083400,0.578706,0.618721,0.721036,0.627182,280924
8,Known delta,0.584592,0.584592,NaN,0.419503,0.697192,0.084392,0.581534,0.620228,0.728340,0.633840,280924
9,Known normalized_delta,0.587239,0.587239,NaN,0.415978,0.698815,0.083086,0.573642,0.626234,0.732677,0.635904,280924
47,Diverse median blend,0.689242,0.578637,0.738352,0.518711,0.766519,0.056289,0.698742,0.738012,0.801330,0.722633,842834
39,Dual | Known conservative blend | Masked 11 blend,0.690335,0.578637,0.739881,0.520002,0.768729,0.057137,0.700509,0.739912,0.803866,0.724217,842834
46,Stable simplex blend,0.690335,0.578637,0.739881,0.520002,0.768729,0.057137,0.700509,0.739912,0.803866,0.724217,842834
43,Dual | Known conservative blend | Spatial clim...,0.693908,0.578637,0.744878,0.521578,0.767450,0.053474,0.700369,0.745582,0.801174,0.725662,842834
0,Notebook 11 public-best,0.692625,0.586791,0.739881,0.522616,0.769866,0.056282,0.703452,0.744714,0.805396,0.726006,842834
18,Dual | Known absolute LGBM | Masked 11 blend,0.692625,0.586791,0.739881,0.522616,0.769866,0.056282,0.703452,0.744714,0.805396,0.726006,842834
25,Dual | delta | Masked 11 blend,0.692005,0.584592,0.739881,0.521245,0.771062,0.057616,0.701297,0.740330,0.805452,0.726173,842834



### Final rapor

1. **Absolute mı delta mı?** Known absolute RMSE `0.586791`, delta RMSE `0.584592`; düşük olan daha iyidir. Normalized delta ve OOF alpha tablolarda ayrıca gösterildi.
2. **Persistence:** Known satırlarda RMSE `0.626942`; gerçek bir zaman-serisi referansıdır.
3. **Climatology:** Global, latitude-band, spatial-cell, location mean/median/monthly sonuçları ayrı hesaplandı; baseline tablosunda doğrudan karşılaştırılabilir.
4. **Known ve masked farkı:** Notebook 11 known `0.586791`, masked `0.739881`; masked problem belirgin biçimde daha zordur.
5. **Spatial katkı:** En iyi spatial deney `Dual | Known conservative blend | Spatial climatology + ML residual` / `0.693908`.
6. **Residual modelling:** En iyi residual deney `Dual | Known conservative blend | Spatial climatology + ML residual` / `0.693908`.
7. **Regime experts:** En iyi regime deney `Dual | Known conservative blend | Global + regime` / `0.693131`; yalnız sabit seçim kuralını geçerse kullanıldı.
8. **Recursive forecasting:** Known RMSE `1.175584`. Teacher forcing yoktur; kendi tahmini ileri taşındı. Baseline'dan kötü ise güvenilir sayılmaz.
9. **En düşük ortalama RMSE:** `Known conservative blend` / `0.578637`.
10. **En iyi robust score:** `Known conservative blend` / `0.627182`.
11. **Public genelleme adayı:** `Dual | Known conservative blend | Masked 11 blend`; yalnız önceden sabit overall/worst/std/masked kurallarından geçtiyse primary seçildi.
12. **Private için iki aday:** Üretilen dosyalar `['Primary']`. Korelasyon `>0.995` ise gereksiz ikinci dosya yazılmadı.
13. **Notebook 11 neden güçlü?** Known/masked routing, missing-aware nonlinear model ve causal climatology blend'i farklı rejimlerde düşük varyans sağlıyor. Yeni model sabit kuralı geçmediyse 11 korunur.
14. **Local→public beklentisi:** Seçim yalnız ortalama OOF'a değil worst-fold, fold std, recent/dry/wet streslerine ve sabit robust score'a bağlıdır. Bu garanti değildir; public skor tuning amacıyla kullanılmadı.

**Pilot/ölçek notu:** KNN/GPR/yoğun robust modeller 2.15M satırda uygulanabilir değildir. `14`te full-validation pilotu LightGBM'i geçmeyen Extra Trees/Random Forest tekrar full-data ana aday yapılmadı. CatBoost kurululuğu raporlandı; geniş arama yapılmadı. Spatial bellek etkisi çözünürlükle oluşan hücre sayısı üzerinden aşağıda raporlanır.


20 degrees cells 96 feature memory MB 8.217063903808594


10 degrees cells 282 feature memory MB 8.217063903808594


5 degrees cells 893 feature memory MB 8.217063903808594


## 11. Minimal continuation — A/B/C

Bu bölüm önceki başarılı hücrelerin çıktılarını koruyarak bağımsız çalışır; notebook baştan yürütülmez. Yalnız üç yaklaşım değerlendirilir:

- A: Notebook 11 baseline
- B: known delta LightGBM + masked absolute LightGBM
- C: A ve B'nin OOF conservative blend'i

Her aşama `artifacts/nb15_minimal/` altında checkpoint oluşturur. Rolling target feature'ları yalnız training geçmişinden gelen 3, 6 ve 12 dönem delta ortalamalarıdır. Spatial-neighbour, recursive forecasting, alternatif model aileleri ve hyperparameter search yoktur.

In [1]:
from pathlib import Path
from time import perf_counter
import numpy as np, pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

TOTAL_START=perf_counter()
SEED=42
ROOT=Path.cwd().resolve()
if not (ROOT/"data/raw/Train.csv").exists(): ROOT=ROOT.parent
DATA=ROOT/"data/raw"; OUT=ROOT/"submissions"; CKPT=ROOT/"artifacts/nb15_minimal"
OUT.mkdir(exist_ok=True); CKPT.mkdir(parents=True,exist_ok=True)
def section(name,start=True,t0=None):
    if start:
        now=perf_counter(); print(f"[START] {name} | elapsed total={now-TOTAL_START:.1f}s"); return now
    now=perf_counter(); print(f"[END] {name} | section={now-t0:.1f}s | elapsed total={now-TOTAL_START:.1f}s")

t0=section("Load + float32 features")
train=pd.read_csv(DATA/"Train.csv",parse_dates=["time"]); test=pd.read_csv(DATA/"Test.csv",parse_dates=["time"])
sample=pd.read_csv(DATA/"SampleSubmission.csv")
SPEI=["SPEI_01_t","SPEI_03_t","SPEI_06_t","SPEI_12_t"]
def features(df):
    o=df.copy(); dt=o.time
    o["month"]=dt.dt.month.astype("int8"); o["quarter"]=dt.dt.quarter.astype("int8"); o["year"]=dt.dt.year.astype("int16")
    o["elapsed_month"]=((dt.dt.year-train.time.dt.year.min())*12+dt.dt.month-1).astype("int16")
    o["month_sin_calc"]=np.sin(2*np.pi*o.month/12).astype("float32"); o["month_cos_calc"]=np.cos(2*np.pi*o.month/12).astype("float32")
    o["season_sin"]=np.sin(2*np.pi*o.quarter/4).astype("float32"); o["season_cos"]=np.cos(2*np.pi*o.quarter/4).astype("float32")
    s=o[SPEI].astype("float32"); o["spei_mean"]=s.mean(axis=1); o["spei_min"]=s.min(axis=1); o["spei_max"]=s.max(axis=1); o["spei_std"]=s.std(axis=1)
    o["spei_01_minus_12"]=(o.SPEI_01_t-o.SPEI_12_t).astype("float32"); o["spei_03_minus_12"]=(o.SPEI_03_t-o.SPEI_12_t).astype("float32")
    o["spei_short_long_ratio"]=(o.SPEI_01_t/(o.SPEI_12_t.abs()+.1)).astype("float32")
    o["lat_band_5"]=np.floor((o.lat+90)/5).astype("int16"); o["lon_band_5"]=np.floor((o.lon+180)/5).astype("int16")
    o["spatial_bin_10"]=(o.lat_band_5*100+o.lon_band_5).astype("int16")
    o["lat_lon_interaction"]=(o.lat*o.lon/1000).astype("float32")
    for c,v in [("lat_sin",np.sin(np.radians(o.lat))),("lat_cos",np.cos(np.radians(o.lat))),
                ("lon_sin",np.sin(np.radians(o.lon))),("lon_cos",np.cos(np.radians(o.lon)))]:o[c]=v.astype("float32")
    return o
trf=features(train); tef=features(test)
ALL=["lat","lon","TWS_t",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year","quarter","elapsed_month",
 "month_sin_calc","month_cos_calc","season_sin","season_cos","spei_mean","spei_min","spei_max","spei_std",
 "spei_01_minus_12","spei_03_minus_12","spei_short_long_ratio","lat_band_5","lon_band_5","spatial_bin_10",
 "lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
MASKED=[c for c in ALL if c!="TWS_t"]
for d in [trf,tef]:
    for c in ALL:
        if c in d and d[c].dtype=="float64": d[c]=d[c].astype("float32")
date_stats=test.groupby("time").agg(n=("ID","size"),known=("TWS_t","count"))
KNOWN_POS=set(np.flatnonzero(date_stats.known.to_numpy()>date_stats.n.to_numpy()*.5))
dates=np.array(sorted(trf.time.unique())); start=len(dates)-54; folds=[]
for i in range(3):
    vd=dates[start+i*18:start+(i+1)*18]; td=dates[:start+i*18]
    ti=np.flatnonzero(trf.time.isin(td).to_numpy()); vi=np.flatnonzero(trf.time.isin(vd).to_numpy())
    vdates=np.array(sorted(trf.iloc[vi].time.unique())); known=trf.iloc[vi].time.isin({d for j,d in enumerate(vdates) if j in KNOWN_POS}).to_numpy()
    folds.append((i+1,ti,vi,known))
section("Load + float32 features",False,t0)

P=dict()
def params(): return dict(n_estimators=160,learning_rate=.08,num_leaves=63,min_child_samples=100,reg_lambda=1,
    subsample=.85,colsample_bytree=.9,random_state=SEED,n_jobs=4,verbosity=-1)
def causal_prior(df):
    o=df.sort_values(["lat","lon","time"]).copy(); sh=o.groupby(["lat","lon"],sort=False).target.shift(1)
    o["prior_location_mean"]=sh.groupby([o.lat,o.lon]).expanding().mean().reset_index(level=[0,1],drop=True)
    o["prior_location_std"]=sh.groupby([o.lat,o.lon]).expanding().std().reset_index(level=[0,1],drop=True)
    o["prior_location_month"]=sh.groupby([o.lat,o.lon,o.month]).expanding().mean().reset_index(level=[0,1,2],drop=True)
    o["prior_global_month"]=o.groupby("month").target.transform(lambda z:z.shift(1).expanding().mean())
    return o.sort_index()
PRI=["prior_location_mean","prior_location_std","prior_location_month","prior_global_month"]
def apply_prior(h,a):
    loc=h.groupby(["lat","lon"]).target.agg(["mean","std"]).rename(columns={"mean":"prior_location_mean","std":"prior_location_std"})
    lm=h.groupby(["lat","lon","month"]).target.agg(["sum","count"]).rename(columns={"sum":"s","count":"n"})
    gm=h.groupby("month").target.mean().rename("prior_global_month")
    o=a.join(loc,on=["lat","lon"]).join(lm,on=["lat","lon","month"]).join(gm,on="month")
    o["prior_location_month"]=(o.s+4*o.prior_location_mean)/(o.n.fillna(0)+4)
    return o.drop(columns=["s","n"])

t0=section("A — Notebook 11 baseline")
a_rows=[]; base_mask_parts=[]; known_abs_parts=[]
for fold,ti,vi,known in folds:
    tr=trf.iloc[ti]; va=trf.iloc[vi]
    km=LGBMRegressor(**params()).fit(tr[ALL],tr.target); kp=km.predict(va.loc[known,ALL])
    mm=LGBMRegressor(**params()).fit(tr[MASKED],tr.target); mp=mm.predict(va.loc[~known,MASKED])
    th=causal_prior(tr); vh=apply_prior(tr,va)
    hm=LGBMRegressor(**params()).fit(th[MASKED+PRI],th.target); hp=hm.predict(vh.loc[~known,MASKED+PRI])
    masked=.65*mp+.35*hp; pred=np.empty(len(va),dtype="float32"); pred[known]=kp; pred[~known]=masked
    a_rows.append(pd.DataFrame({"fold":fold,"target":va.target.to_numpy("float32"),"pred":pred,"known":known}))
    base_mask_parts.append(masked.astype("float32")); known_abs_parts.append(kp.astype("float32"))
A=pd.concat(a_rows,ignore_index=True)
np.savez_compressed(CKPT/"A_baseline_oof.npz",fold=A.fold.to_numpy("int8"),target=A.target.to_numpy("float32"),pred=A.pred.to_numpy("float32"),known=A.known.to_numpy(bool))
section("A — Notebook 11 baseline",False,t0)

def delta_history_train(df):
    o=df.sort_values(["lat","lon","time"]).copy(); delta=(o.target-o.TWS_t).astype("float32")
    shifted=delta.groupby([o.lat,o.lon]).shift(1)
    for w in [3,6,12]:
        o[f"delta_roll{w}"]=shifted.groupby([o.lat,o.lon]).rolling(w,min_periods=1).mean().reset_index(level=[0,1],drop=True).astype("float32")
    return o.sort_index()
def delta_history_apply(h,a):
    o=a.copy(); hs=h.sort_values(["lat","lon","time"]).copy(); hs["_delta"]=(hs.target-hs.TWS_t).astype("float32")
    for w in [3,6,12]:
        stat=hs.groupby(["lat","lon"],group_keys=False).tail(w).groupby(["lat","lon"])._delta.mean().rename(f"delta_roll{w}")
        o=o.join(stat,on=["lat","lon"])
    return o
DROLL=["delta_roll3","delta_roll6","delta_roll12"]

t0=section("B — known delta + masked absolute")
b_rows=[]
for j,(fold,ti,vi,known) in enumerate(folds):
    tr=trf.iloc[ti]; va=trf.iloc[vi]; th=delta_history_train(tr); vh=delta_history_apply(tr,va)
    use=th.TWS_t.notna(); dm=LGBMRegressor(**params()).fit(th.loc[use,ALL+DROLL],(th.loc[use].target-th.loc[use].TWS_t).astype("float32"))
    kp=va.loc[known].TWS_t.to_numpy("float32")+dm.predict(vh.loc[known,ALL+DROLL]).astype("float32")
    pred=np.empty(len(va),dtype="float32"); pred[known]=kp; pred[~known]=base_mask_parts[j]
    b_rows.append(pd.DataFrame({"fold":fold,"target":va.target.to_numpy("float32"),"pred":pred,"known":known}))
B=pd.concat(b_rows,ignore_index=True)
np.savez_compressed(CKPT/"B_delta_dual_oof.npz",fold=B.fold.to_numpy("int8"),target=B.target.to_numpy("float32"),pred=B.pred.to_numpy("float32"),known=B.known.to_numpy(bool))
section("B — known delta + masked absolute",False,t0)

t0=section("C — conservative OOF blend")
blend_rows=[]
for w in np.arange(0,1.01,.1):
    p=((1-w)*A.pred+w*B.pred).to_numpy("float32"); fs=[root_mean_squared_error(A.loc[A.fold==f,"target"],p[A.fold.to_numpy()==f]) for f in [1,2,3]]
    blend_rows.append((float(w),root_mean_squared_error(A.target,p),max(fs),np.std(fs)))
blend_grid=pd.DataFrame(blend_rows,columns=["B_weight","Overall RMSE","Worst-fold RMSE","Fold std"])
W=float(blend_grid.sort_values(["Overall RMSE","Worst-fold RMSE"]).iloc[0].B_weight)
C=A.copy(); C["pred"]=((1-W)*A.pred+W*B.pred).astype("float32")
np.savez_compressed(CKPT/"C_conservative_oof.npz",fold=C.fold.to_numpy("int8"),target=C.target.to_numpy("float32"),pred=C.pred.to_numpy("float32"),known=C.known.to_numpy(bool),B_weight=np.float32(W))
section("C — conservative OOF blend",False,t0)

def metrics(name,z):
    fs=z.groupby("fold").apply(lambda q:root_mean_squared_error(q.target,q.pred),include_groups=False)
    k=z[z.known]; m=z[~z.known]
    return {"Model":name,"Overall RMSE":root_mean_squared_error(z.target,z.pred),"Known RMSE":root_mean_squared_error(k.target,k.pred),
      "Masked RMSE":root_mean_squared_error(m.target,m.pred),"MAE":mean_absolute_error(z.target,z.pred),
      "Worst-fold RMSE":fs.max(),"Fold std":fs.std(ddof=0)}
table=pd.DataFrame([metrics("A — Notebook 11 baseline",A),metrics("B — delta dual expert",B),metrics(f"C — conservative blend (B weight={W:.1f})",C)])
base=table.iloc[0]; valid=table[(table["Overall RMSE"]<base["Overall RMSE"]) & (table["Worst-fold RMSE"]<=base["Worst-fold RMSE"]+.001)]
chosen=(valid.sort_values("Overall RMSE").iloc[0] if len(valid) else base)
CHOSEN_NAME=chosen.Model; print(table.to_string(index=False)); print("CHOSEN",CHOSEN_NAME)
pd.DataFrame(table).to_csv(CKPT/"comparison.csv",index=False)

t0=section("Full fit + validated submission")
known_test=tef.TWS_t.notna().to_numpy()
km=LGBMRegressor(**params()).fit(trf[ALL],trf.target); known_a=km.predict(tef.loc[known_test,ALL]).astype("float32")
mm=LGBMRegressor(**params()).fit(trf[MASKED],trf.target); masked_abs=mm.predict(tef.loc[~known_test,MASKED]).astype("float32")
th=causal_prior(trf); vh=apply_prior(trf,tef); hm=LGBMRegressor(**params()).fit(th[MASKED+PRI],th.target)
masked_hist=hm.predict(vh.loc[~known_test,MASKED+PRI]).astype("float32"); masked_a=.65*masked_abs+.35*masked_hist
pred_a=np.empty(len(tef),dtype="float32"); pred_a[known_test]=known_a; pred_a[~known_test]=masked_a
dh=delta_history_train(trf); dv=delta_history_apply(trf,tef); use=dh.TWS_t.notna()
dm=LGBMRegressor(**params()).fit(dh.loc[use,ALL+DROLL],(dh.loc[use].target-dh.loc[use].TWS_t).astype("float32"))
known_b=tef.loc[known_test].TWS_t.to_numpy("float32")+dm.predict(dv.loc[known_test,ALL+DROLL]).astype("float32")
pred_b=np.empty(len(tef),dtype="float32"); pred_b[known_test]=known_b; pred_b[~known_test]=masked_abs
pred_c=((1-W)*pred_a+W*pred_b).astype("float32")
pred={"A — Notebook 11 baseline":pred_a,"B — delta dual expert":pred_b,f"C — conservative blend (B weight={W:.1f})":pred_c}[CHOSEN_NAME]
byid=pd.Series(pred,index=tef.ID.astype(str)); sub=sample.copy(); sub["Target"]=sample.ID.astype(str).map(byid).to_numpy("float32")
checks={"rows":len(sub)==280961,"columns":list(sub)==["ID","Target"],"id_set":set(sub.ID.astype(str))==set(tef.ID.astype(str)),
 "unique":sub.ID.is_unique,"no_nan":sub.Target.notna().all(),"finite":np.isfinite(sub.Target).all(),
 "numeric":pd.api.types.is_numeric_dtype(sub.Target),"order":sub.ID.equals(sample.ID),
 "known_rows":int(known_test.sum()),"masked_rows":int((~known_test).sum())}
assert all(v for v in checks.values() if isinstance(v,(bool,np.bool_))),checks
path=OUT/"submission_15_primary.csv"; sub.to_csv(path,index=False); reread=pd.read_csv(path)
assert len(reread)==280961 and reread.ID.equals(sample.ID) and np.isfinite(reread.Target).all()
pd.DataFrame([{"chosen":CHOSEN_NAME,"B_weight":W,**checks,"pred_min":sub.Target.min(),"pred_max":sub.Target.max(),
 "pred_mean":sub.Target.mean(),"pred_std":sub.Target.std()}]).to_csv(CKPT/"submission_report.csv",index=False)
section("Full fit + validated submission",False,t0)
print("SUBMISSION",path); print(checks); print("range",float(sub.Target.min()),float(sub.Target.max()))
display(table)
display(blend_grid)
print(f"TOTAL ELAPSED {perf_counter()-TOTAL_START:.1f}s")


[START] Load + float32 features | elapsed total=0.0s


[END] Load + float32 features | section=18.3s | elapsed total=18.3s
[START] A — Notebook 11 baseline | elapsed total=18.3s


[END] A — Notebook 11 baseline | section=191.7s | elapsed total=210.0s
[START] B — known delta + masked absolute | elapsed total=210.0s


[END] B — known delta + masked absolute | section=77.5s | elapsed total=287.4s
[START] C — conservative OOF blend | elapsed total=287.4s


[END] C — conservative OOF blend | section=1.7s | elapsed total=289.1s


                                Model  Overall RMSE  Known RMSE  Masked RMSE      MAE  Worst-fold RMSE  Fold std
             A — Notebook 11 baseline      0.695429    0.587902     0.743377 0.524914         0.772187  0.055909
                B — delta dual expert      0.696090    0.590245     0.743377 0.525273         0.775337  0.057745
C — conservative blend (B weight=0.5)      0.694090    0.583135     0.743377 0.523424         0.772587  0.057191
CHOSEN C — conservative blend (B weight=0.5)
[START] Full fit + validated submission | elapsed total=289.6s


[END] Full fit + validated submission | section=115.8s | elapsed total=405.4s
SUBMISSION C:\global-water-storage-forecasting\submissions\submission_15_primary.csv
{'rows': True, 'columns': True, 'id_set': True, 'unique': True, 'no_nan': np.True_, 'finite': np.True_, 'numeric': True, 'order': True, 'known_rows': 94048, 'masked_rows': 186913}
range -2.3291115760803223 2.123260021209717


,Model,Overall RMSE,Known RMSE,Masked RMSE,MAE,Worst-fold RMSE,Fold std
0,A — Notebook 11 baseline,0.695429,0.587902,0.743377,0.524914,0.772187,0.055909
1,B — delta dual expert,0.696090,0.590245,0.743377,0.525273,0.775337,0.057745
2,C — conservative blend (B weight=0.5),0.694090,0.583135,0.743377,0.523424,0.772587,0.057191


,B_weight,Overall RMSE,Worst-fold RMSE,Fold std
0,0.0,0.695429,0.772187,0.055909
1,0.1,0.694894,0.772079,0.056221
2,0.2,0.694493,0.772065,0.056506
3,0.3,0.694225,0.772145,0.056763
4,0.4,0.694090,0.772319,0.056991
5,0.5,0.694090,0.772587,0.057191
6,0.6,0.694223,0.772950,0.057361
7,0.7,0.694490,0.773406,0.057502
8,0.8,0.694890,0.773956,0.057613
9,0.9,0.695424,0.774600,0.057694


TOTAL ELAPSED 405.4s
